# AdAHuman — patch optimization on Colab

Optimizes the person-suppression patch (RQ1b) on a GPU. Everything else in
the artifact runs locally on CPU.

**What crosses the boundary.** Only the source bundle goes up, and only the
patch tensor plus its training log and run record come back. Colab never sees
the held-out evaluation pool: `train_patch` is entitled to read `attack_dev`
and `reference` only, and `PoolDataset` refuses to construct against anything
else. The patch is therefore a frozen input to local evaluation, and the
training device never enters a reported measurement.

**Before running:** Runtime → Change runtime type → GPU.

This notebook uses no interactive upload or download widgets, so it behaves
the same in the Colab web UI and in a VS Code notebook client. Files move via
the filesystem: put the bundle somewhere the next cell searches, and collect
outputs from the directory the last cell reports.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## 2. Locate and extract the source bundle

Generate it locally first:

```bash
bash scripts/make_colab_bundle.sh
```

The next cell searches for `adahuman_colab_src.tar.gz` in the current
directory, its parent directories (the notebook lives in `notebooks/`, so the
repo root is a parent), and the usual Colab locations.

**On a hosted Colab runtime**, a file sitting on your laptop is not visible to
the VM at all. It has to get there first — either drop it in Google Drive and
set `MOUNT_DRIVE = True`, or upload it into `/content` with the Files pane in
the left sidebar.

The cell prints the bundle's sha256. Record it — it identifies the exact code
that produced the patch.

In [ ]:
import hashlib, os, pathlib, tarfile

BUNDLE_NAME = 'adahuman_colab_src.tar.gz'
MOUNT_DRIVE = False   # True to mount Google Drive and search it

# A hosted Colab VM has /content; a local runtime does not. This decides both
# where to search and where it is safe to extract.
ON_COLAB_VM = pathlib.Path('/content').is_dir()

if MOUNT_DRIVE and ON_COLAB_VM and not pathlib.Path('/content/drive/MyDrive').is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Drive not mounted ({exc}); searching other locations')

here = pathlib.Path.cwd().resolve()

# Ancestors matter: run from notebooks/, the repo root is one level up. Capped
# at three levels so the search cannot wander toward the filesystem root.
CANDIDATE_DIRS = [here, *list(here.parents)[:3]]
CANDIDATE_DIRS += [
    pathlib.Path('/content'),
    pathlib.Path('/content/drive/MyDrive'),
    pathlib.Path('/content/drive/MyDrive/AdAHuman'),
    pathlib.Path.home(),
]

def find_bundle():
    """Exact matches first, then one level down. Never a recursive walk --
    globbing '**' under a home directory can take minutes."""
    for directory in CANDIDATE_DIRS:
        candidate = directory / BUNDLE_NAME
        if candidate.is_file():
            return candidate
    for directory in CANDIDATE_DIRS:
        if directory.is_dir():
            for match in sorted(directory.glob(f'*/{BUNDLE_NAME}')):
                return match
    return None

bundle = find_bundle()

if bundle is None:
    print(f'{BUNDLE_NAME} not found.\n')
    print(f'cwd:         {here}')
    print(f'colab VM:    {ON_COLAB_VM}')
    print('\nsearched:')
    for directory in CANDIDATE_DIRS:
        print(f'  [{"y" if directory.is_dir() else "-"}] {directory}')

    nearby = [p for d in CANDIDATE_DIRS if d.is_dir() for p in d.glob('*.tar.gz')]
    if nearby:
        print('\ntar.gz files that are visible:')
        for path in nearby:
            print(f'  {path}')

    if ON_COLAB_VM:
        hint = ('You are on a hosted Colab VM. A file on your laptop is not '
                'visible here. Upload it into /content using the Files pane, '
                'or put it in Drive and set MOUNT_DRIVE = True.')
    else:
        hint = ('Run `bash scripts/make_colab_bundle.sh` from the repo root, '
                'or append the bundle\'s directory to CANDIDATE_DIRS.')
    raise FileNotFoundError(hint)

digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
print(f'bundle  {bundle}')
print(f'size    {bundle.stat().st_size / 1024:.0f} KiB')
print(f'sha256  {digest}')

# Extract beside the bundle on a local runtime; /content on a Colab VM.
WORKDIR = pathlib.Path('/content/adahuman') if ON_COLAB_VM else bundle.parent
WORKDIR.mkdir(parents=True, exist_ok=True)
with tarfile.open(bundle) as archive:
    archive.extractall(WORKDIR)
os.chdir(WORKDIR)

print(f'\nextracted to {WORKDIR}')
print(sorted(p.name for p in WORKDIR.iterdir() if not p.name.startswith('.')))

## 3. Dependencies

Colab ships its own torch build, which is kept: reinstalling the pinned CPU
versions here would discard CUDA support. The version actually used is
captured in the run log, so the difference from the local pin is recorded
rather than hidden.

In [ ]:
!pip install -q pycocotools scikit-learn PyYAML 2>&1 | tail -2
import torch, torchvision
print('torch      ', torch.__version__)
print('torchvision', torchvision.__version__)

## 4. Fetch COCO val2017

In [ ]:
!bash scripts/00_fetch_coco.sh

## 5. Verify the pools match the frozen manifests

The manifests came up in the bundle. This re-derives pool membership from the
seed and checks it against them, confirming that the Colab environment selects
the same images the local freeze did.

In [ ]:
!python scripts/02_freeze_pools.py

## 6. Timing probe

Five steps, then an estimate of what each epoch budget costs. Use it to pick
the epoch count below. Writes nothing.

In [ ]:
!python scripts/04_train_patch.py --probe-timing --workers 2

## 7. Train

Watch `max-person`: it starts near 0.83 and should fall. If it plateaus well
above the 0.5 decision threshold, the attack is weak — report that rather than
extending the run, and note it in `LIMITATIONS.md`. Extending until the number
looks good is how a schedule turns into a result.

`--write-steps` freezes `attack.steps` in the protocol on completion.

In [ ]:
!python scripts/04_train_patch.py --epochs 30 --workers 2 --write-steps

## 8. Collect the outputs

Copies the results to Drive when it is mounted, so they survive the session
and can be retrieved without a download widget. Falls back to leaving them in
the session and printing their paths.

Place `patch_v1.*` in `artifacts/` and the run log in `logs/` in the local
repo, and replace `configs/protocol_v1.yaml` with the copy returned here —
`--write-steps` modified it. The run log is what ties the patch to the code,
the seed, and the GPU that produced it.

In [ ]:
import glob, hashlib, pathlib, shutil

outputs = (
    sorted(glob.glob('artifacts/patch_v1*'))
    + sorted(glob.glob('logs/*train_patch.json'))
    + ['configs/protocol_v1.yaml']
)
outputs = [p for p in outputs if pathlib.Path(p).is_file()]
if not outputs:
    raise FileNotFoundError('no outputs found; did the training cell finish?')

drive_root = pathlib.Path('/content/drive/MyDrive')
destination = drive_root / 'AdAHuman_results' if drive_root.is_dir() else None
if destination:
    destination.mkdir(parents=True, exist_ok=True)

print(f'{"sha256":18s}  {"size":>9s}  file')
for path in outputs:
    source = pathlib.Path(path)
    digest = hashlib.sha256(source.read_bytes()).hexdigest()
    print(f'{digest[:16]}    {source.stat().st_size:9,d}  {path}')
    if destination:
        shutil.copy2(source, destination / source.name)

if destination:
    print(f'\ncopied {len(outputs)} files to {destination}')
    print('Retrieve them from Google Drive, then commit them in the local repo.')
else:
    print(f'\nDrive is not mounted. Files remain in {pathlib.Path.cwd()}.')
    print('Download them via the Files pane, or set MOUNT_DRIVE = True in cell 2')
    print('and rerun this cell.')

## 9. Preview the patch

A sanity check, not a result. A patch that is uniform or saturated usually
means the optimizer diverged.

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/patch_v1.png'))